# quantum-cortex — reproduce the Molaison dissociation in minutes, without a GPU

The Molaison dissociation (the H.M. protocol: named after Henry Molaison, the patient whose hippocampus was removed in 1953 and who formed no new episodic memory from that day while keeping every skill he had) asks a memory the same question: with its journal cut, do the episodes vanish while the skills stay? The specification and its thresholds were frozen on 2026-08-08, before any result (`docs/benchmarks/hm-protocol.md`).

This notebook runs the persistent variant against the C2b organ: 200 leakage proof synthetic facts planted through the gated write path into a journal sealed on disk, then a session B that reopens the journal from the disk alone (a new object, the index recomputed) and probes recall with the journal on and off, plus a 50 fact negative control. Four seeds, each with its own generator hash. **Settings:** Accelerator = None, Internet = On. About a minute per seed on a CPU.

The verdict is `hm_dissociation_pass` with `persistent: true` and `claimable: 1`; the files it writes are the artefacts (`metrics/mqar/hm-protocol-persistent-*.json`). Read them, not the console.

In [ ]:
!rm -rf quantum-cortex
!git clone https://github.com/vaneeckhoutnicolas/quantum-cortex.git
%cd quantum-cortex
!pip -q install numpy cryptography jsonschema


In [ ]:
import json, subprocess, sys
rows = []
for seed in (0, 1, 2, 3):
    out = f'metrics/mqar/hm-protocol-persistent-reproduced-seed{seed}.json'
    subprocess.run([sys.executable, '-m', 'cortex_c2b.hm_protocol', '--persistent', '--seed', str(seed), '--out', out], check=True, capture_output=True)
    r = json.load(open(out))
    rows.append((seed, r['hm_recall_on'], r['hm_recall_off'], r['hm_gap'], r['hm_negctrl_rate'], r['hm_skill_delta'], r['hm_dissociation_pass'], r['persistent'], r['claimable'], r['generator']['config_hash']))
print('seed | recall on | recall off | gap | negative control | skill delta | pass | persistent | claimable | generator')
for row in rows: print(' | '.join(str(x) for x in row))
print()
print('thresholds (frozen 2026-08-08):', r['thresholds'])


If the four lines read `pass 1`, `persistent True`, `claimable 1`, you have reproduced RESULTS row 13b. The same protocol's language model arm (the model reading its own journal in generation) is measured separately, and its results, INVALID and attributed run after run, are in RESULTS rows 21 onward: the benchmark is credible because it knows how to fail.